# CNN (TensorFlow)

Thesis experiment: two Conv1D layers, rolling-origin evaluation, three feature sets. Needs `tensorflow`.

In [ ]:
import pandas as pd
import numpy as np
from math import sqrt
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from keras.layers import Conv1D, MaxPooling1D, Flatten, Dense
from keras.models import Sequential

In [ ]:
from energyforecast.data import load_dataset

data = load_dataset("../../data")

In [ ]:
def plot_model_rmse_and_loss(history):

    #evaluating train and validation accuracies and losses

    train_rmse = history.history['root_mean_squared_error']
    #val_rmse = history.history['val_root_mean_squared_error']

    train_loss = history.history['loss']
    #val_loss = history.history['val_loss']

    #visualizing epochs vs. train and validation accuracies and losses

    plt.figure(figsize=(20, 10))
    plt.subplot(1, 2, 1)
    plt.plot(train_rmse, label='Training RMSE')
    #plt.plot(val_rmse, label='Validation RMSE')
    plt.legend()
    plt.title('Epochs vs. Training and Validation RMSE')

    plt.subplot(1, 2, 2)
    plt.plot(train_loss, label='Training Loss')
    #plt.plot(val_loss, label='Validation Loss')
    plt.legend()
    plt.title('Epochs vs. Training and Validation Loss')

    plt.show()

def plot_preds_vs_actual(true, preds):
    plt.figure(figsize=(12,6))
    plt.plot(true, label='Real')
    plt.plot(preds, label='Predicted', color='red')
    plt.title('Predicted vs Real Values')
    plt.title('Actual vs Predicted Values')
    plt.xlabel('Time')
    plt.ylabel('Total Aggregated')
    plt.legend()
    plt.show()

In [ ]:
def custom_train_test(train, test, steps):
    X_train, y_train, X_test, y_test = [], [], [], []
    for i in range(steps, len(train)):
        X_train.append(train[i-steps:i])
        y_train.append(train[i])

    for i in range(steps, len(test)):
        X_test.append(test[i-steps:i])
        y_test.append(test[i])

    X_train, y_train = np.array(X_train), np.array(y_train)
    X_test, y_test = np.array(X_test), np.array(y_test)

    return X_train, y_train, X_test, y_test

n_train = 35064
n_test = int(len(data)-n_train)
window_size = 24

features = ['total_aggregated']
feature_array = data[features].values

# Fit Scaler only on Training target values
target_scaler = StandardScaler()
target_scaler.fit(feature_array[:n_train].reshape(-1,1))

# Transform both Training and Test data
scaled_array = target_scaler.transform(feature_array)

# Define window size for input data

train_data = scaled_array[:n_train]
test_data = scaled_array[len(train_data):]

X_train, y_train, X_test, y_test = custom_train_test(train_data, test_data, 2)

In [ ]:
truth = feature_array[-len(y_test):]

In [ ]:
loss = tf.keras.losses.MeanSquaredError()
metric = [tf.keras.metrics.RootMeanSquaredError()]
lr_schedule = tf.keras.callbacks.LearningRateScheduler(
              lambda epoch: 1e-4 * 10**(epoch / 10))
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='loss',patience=3)
optimizer = tf.keras.optimizers.legacy.Adam(learning_rate=0.01)

In [ ]:
history_X = X_train.copy()
history_y = y_train.copy()

In [ ]:
input_shape1 = X_train.shape[-2:]
input_shape1

In [ ]:
n_forecast_steps = 24  # forecast horizon
window = 24*365*3

In [ ]:
# Define the CNN model
cnn = Sequential()
cnn.add(Conv1D(filters=16, kernel_size=1, activation='relu', input_shape=input_shape1))
cnn.add(Conv1D(filters=16, kernel_size=1, activation='relu'))
cnn.add(Flatten())
cnn.add(Dense(1))
cnn.compile(optimizer=optimizer, loss=loss, metrics=metric)

cnn.summary()

In [ ]:
history_X = X_train.copy()  # history starts as the training window
history_y = y_train.copy()

predictions = []  # forecasts
errors = []  # RMSE per period
days = 1  # period counter
for i in range(0, len(X_test), n_forecast_steps):
    t_end = i + n_forecast_steps  # forecast the block [i, i + n_forecast_steps)

    if t_end > len(X_test):
        t_end = len(X_test)
    X_test_block = X_test[i:t_end]
    y_test_block = y_test[i:t_end]
    truth_block = truth[i:t_end]

    print('training and predicting for period', days)
    cnn.fit(history_X, history_y, epochs=100, verbose=0, batch_size=168, callbacks=[early_stopping])  # fit on the current history, up to 100 epochs
    # early stopping ends the fit when the loss
    # stops decreasing for `patience` epochs

    yhat = cnn.predict(X_test_block)  # predict and map back to MW

    yhat_rescaled = target_scaler.inverse_transform(yhat)

    rmse = sqrt(np.mean((truth_block - yhat_rescaled)**2))  # RMSE of this block
    errors.append(rmse)
    predictions.extend(yhat_rescaled)

    print('RMSE:', rmse)

    days = days + 1

    # Update the history with the latest "known" data
    history_X = np.concatenate((history_X, X_test_block), axis=0)
    history_X = history_X[n_forecast_steps:]

    history_y = np.concatenate((history_y, y_test_block), axis=0)
    history_y = history_y[n_forecast_steps:]

In [ ]:
## Plot the actual and predicted values
plot_preds_vs_actual(truth, predictions)
rmse = sqrt(np.mean((truth - predictions)**2))

In [ ]:
res = pd.DataFrame(pd.Series(truth.flatten()), columns=['y_true'])
res['y_pred'] = predictions

In [ ]:
rmse = sqrt(np.mean((res.y_true - res.y_pred)**2))
print('RMSE:', rmse)

## Weekend dummies

In [ ]:
def create_timestamps(vector, window_size):
    X, Y = [], []
    for i in range(window_size, len(vector)):
        X.append(vector[i-window_size:i, :])
        Y.append(vector[i, 0])
    X, Y = np.array(X), np.array(Y)
    return X, Y

In [ ]:
features = ['total_aggregated', 'saturday', 'sunday']
feature_array = data[features].values

# Fit Scaler only on Training target values
target_scaler = StandardScaler()
target_scaler.fit(feature_array[:n_train, 0].reshape(-1,1))

# Transform both Training and Test data
#scaled_array = target_scaler.transform(feature_array)

scaled_feature = target_scaler.transform(feature_array[:, 0].reshape(-1, 1))

scaled_array = np.copy(feature_array)
scaled_array[:, 0] = scaled_feature.flatten()

In [ ]:
# Define window size for input data
train_data = scaled_array[:n_train]
test_data = scaled_array[len(train_data):]

#train_data = feature_array[:n_train]
#test_data = feature_array[len(train_data):]

## Prepare training data
#X_train, y_train = [], []
#for i in range(window_size, len(train_data)):
#    X_train.append(train_data[i-window_size:i, :])
#    y_train.append(train_data[i, 0])
#X_train, y_train = np.array(X_train), np.array(y_train)
#
## Prepare test data
#X_test, y_test = [], []
#for i in range(window_size, len(test_data)):
#    X_test.append(test_data[i-window_size:i, :])
#    y_test.append(test_data[i, 0])
#X_test, y_test = np.array(X_test), np.array(y_test)

X_train, y_train = create_timestamps(train_data, 2)
X_test, y_test = create_timestamps(test_data, 2)

In [ ]:
# Define the CNN model
cnn2 = Sequential()
cnn2.add(Conv1D(filters=16, kernel_size=1, activation='relu', input_shape=(X_train.shape[1], X_train.shape[2])))
cnn2.add(Conv1D(filters=16, kernel_size=1, activation='relu'))
#cnn2.add(MaxPooling1D(pool_size=2))
cnn2.add(Flatten())
#cnn2.add(Dense(1, activation='relu'))
cnn2.add(Dense(1))

cnn2.compile(optimizer=optimizer, loss=loss, metrics=metric)

cnn2.summary()

In [ ]:
history_X = X_train.copy()  # history starts as the training window
history_y = y_train.copy()

predictions2 = []  # forecasts
errors2 = []  # RMSE per period
days = 1  # period counter
for i in range(0, len(X_test), n_forecast_steps):
    t_end = i + n_forecast_steps  # forecast the block [i, i + n_forecast_steps)

    if t_end > len(X_test):
        t_end = len(X_test)
    X_test_block = X_test[i:t_end]
    y_test_block = y_test[i:t_end]
    truth_block = truth[i:t_end]

    print('training and predicting for period', days)
    cnn2.fit(history_X, history_y, epochs=100, verbose=0, batch_size=168, callbacks=[early_stopping])  # fit on the current history, up to 100 epochs
    # early stopping ends the fit when the loss
    # stops decreasing for `patience` epochs

    yhat = cnn2.predict(X_test_block)  # predict and map back to MW

    yhat_rescaled = target_scaler.inverse_transform(yhat)

    rmse = sqrt(np.mean((truth_block - yhat_rescaled)**2))  # RMSE of this block
    errors2.append(rmse)
    predictions2.extend(yhat_rescaled)

    print('RMSE:', rmse)

    days = days + 1

    # Update the history with the latest "known" data
    history_X = np.concatenate((history_X, X_test_block), axis=0)
    history_X = history_X[n_forecast_steps:]

    history_y = np.concatenate((history_y, y_test_block), axis=0)
    history_y = history_y[n_forecast_steps:]

In [ ]:
plot_preds_vs_actual(truth, predictions2)

In [ ]:
rmse2 = sqrt(np.mean((truth - predictions2)**2))

print('RMSE:', rmse2)

#mse_inv = sqrt(np.mean((y_pred_inv - y_test_inv)**2))
#print('Mean Squared Error:', mse_inv)

## Business-hour dummy

In [ ]:
features = ['total_aggregated', 'business_hour']
feature_array = data[features].values

# Fit Scaler only on Training target values
target_scaler = StandardScaler()
target_scaler.fit(feature_array[:n_train, 0].reshape(-1,1))

# Transform both Training and Test data
#scaled_array = target_scaler.transform(feature_array)

scaled_feature = target_scaler.transform(feature_array[:, 0].reshape(-1, 1))

scaled_array = np.copy(feature_array)
scaled_array[:, 0] = scaled_feature.flatten()
# Define window size for input data
train_data = scaled_array[:n_train]
test_data = scaled_array[len(train_data):]

#train_data = feature_array[:n_train]
#test_data = feature_array[len(train_data):]

# Prepare training data
#X_train, y_train = [], []
#for i in range(window_size, len(train_data)):
#    X_train.append(train_data[i-window_size:i, :])
#    y_train.append(train_data[i, 0])
#X_train, y_train = np.array(X_train), np.array(y_train)
#
## Prepare test data
#X_test, y_test = [], []
#for i in range(window_size, len(test_data)):
#    X_test.append(test_data[i-window_size:i, :])
#    y_test.append(test_data[i, 0])
#X_test, y_test = np.array(X_test), np.array(y_test)
X_train, y_train = create_timestamps(train_data, 2)
X_test, y_test = create_timestamps(test_data, 2)

In [ ]:
# Define the CNN model
cnn3 = Sequential()
cnn3.add(Conv1D(filters=16, kernel_size=1, activation='relu', input_shape=(X_train.shape[1], X_train.shape[2])))
cnn3.add(Conv1D(filters=16, kernel_size=1, activation='relu'))
cnn3.add(Flatten())
cnn3.add(Dense(1))
cnn3.compile(optimizer=optimizer, loss=loss, metrics=metric)

cnn3.summary()

In [ ]:
history_X = X_train.copy()  # history starts as the training window
history_y = y_train.copy()

predictions3 = []  # forecasts
errors3 = []  # RMSE per period
days = 1  # period counter
for i in range(0, len(X_test), n_forecast_steps):
    t_end = i + n_forecast_steps  # forecast the block [i, i + n_forecast_steps)

    if t_end > len(X_test):
        t_end = len(X_test)
    X_test_block = X_test[i:t_end]
    y_test_block = y_test[i:t_end]
    truth_block = truth[i:t_end]

    print('training and predicting for period', days)
    cnn3.fit(history_X, history_y, epochs=100, verbose=0, callbacks=[early_stopping])  # fit on the current history, up to 100 epochs
    # early stopping ends the fit when the loss
    # stops decreasing for `patience` epochs

    yhat = cnn3.predict(X_test_block)  # predict and map back to MW

    yhat_rescaled = target_scaler.inverse_transform(yhat)

    rmse = sqrt(np.mean((truth_block - yhat_rescaled)**2))  # RMSE of this block
    errors3.append(rmse)
    predictions3.extend(yhat_rescaled)

    print('RMSE:', rmse)

    days = days + 1

    # Update the history with the latest "known" data
    history_X = np.concatenate((history_X, X_test_block), axis=0)
    history_X = history_X[n_forecast_steps:]

    history_y = np.concatenate((history_y, y_test_block), axis=0)
    history_y = history_y[n_forecast_steps:]

In [ ]:
plot_preds_vs_actual(truth, predictions3)
rmse3 = sqrt(np.mean((truth - predictions3)**2))
print(rmse3)

In [ ]:
res = pd.DataFrame(predictions3, columns=['y_pred3'])
res['y_pred'] = predictions
res['y_pred2'] = predictions2
res['y_true'] = pd.Series(truth.flatten())

In [ ]:
rmse1 = sqrt(np.mean((res.y_true - res.y_pred)**2))
print('Root Mean Squared Error just TS:', rmse1)
rmse2 = sqrt(np.mean((res.y_true - res.y_pred2)**2))
print('Root Mean Squared Error WE dummies:', rmse2)
rmse3 = sqrt(np.mean((res.y_true - res.y_pred3)**2))
print('Root Mean Squared Error BH dummy:', rmse3)